<a href="https://www.kaggle.com/code/muhammaddhiyaulatha/arc-baseline-zero-model-ipynb?scriptVersionId=311482965" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# ARC Prize 2026 - Baseline Model

This notebook contains my first submission to the ARC-AGI-2 competition on Kaggle.

## Approach

* Generate output grids filled with zeros
* Match input grid dimensions
* Ensure correct submission format

## Purpose

This is a baseline to understand:

* Submission pipeline
* Evaluation system
* Dataset structure

Next step: implement rule-based reasoning.


In [2]:
import json
import numpy as np
from collections import Counter

class ARCSolver:
    def __init__(self):
        # Daftar heuristik diperluas
        self.basic_rules = [
            self.identity, self.rotate_90, self.rotate_180, 
            self.rotate_270, self.flip_h, self.flip_v, 
            self.transpose, self.color_shift, self.tile_2x2, 
            self.crop_to_content, self.fill_background
        ]

    # --- HEURISTIK LANJUTAN ---

    def crop_to_content(self, x):
        """Memotong grid hanya pada bagian yang memiliki objek (non-zero)."""
        coords = np.argwhere(x != 0)
        if coords.size == 0: return x
        x_min, y_min = coords.min(axis=0)
        x_max, y_max = coords.max(axis=0)
        return x[x_min:x_max+1, y_min:y_max+1]

    def fill_background(self, x):
        """Mengganti warna background (0) dengan warna paling dominan ke-2."""
        counts = Counter(x.flatten())
        if len(counts) < 2: return x
        # Ambil warna paling umum yang bukan 0
        non_zero_colors = [c for c, _ in counts.most_common() if c != 0]
        if not non_zero_colors: return x
        fill_color = non_zero_colors[0]
        new_grid = x.copy()
        new_grid[x == 0] = fill_color
        return new_grid

    # --- BASIC TRANSFORMATIONS (OPTIMIZED) ---
    def identity(self, x): return x
    def rotate_90(self, x): return np.rot90(x, -1)
    def rotate_180(self, x): return np.rot90(x, -2)
    def rotate_270(self, x): return np.rot90(x, -3)
    def flip_h(self, x): return np.flip(x, axis=1)
    def flip_v(self, x): return np.flip(x, axis=0)
    def transpose(self, x): return x.T
    def tile_2x2(self, x): return np.tile(x, (2, 2))

    def color_shift(self, x):
        counts = Counter(x.flatten())
        if len(counts) < 2: return x
        c = counts.most_common(2)
        c1, c2 = c[0][0], c[1][0]
        res = x.copy()
        res[x == c1], res[x == c2] = c2, c1
        return res

    # --- CORE ENGINE ---

    def check_match(self, pred, target):
        target_arr = np.array(target)
        return pred.shape == target_arr.shape and np.array_equal(pred, target_arr)

    def find_best_rule(self, train_cases):
        # Strategi 1: Single Rule
        for rule in self.basic_rules:
            try:
                if all(self.check_match(rule(np.array(ex['input'])), ex['output']) for ex in train_cases):
                    return rule
            except: continue

        # Strategi 2: Composition (R1 -> R2)
        # Kita batasi pencarian agar tidak timeout di Kaggle
        for r1 in self.basic_rules:
            for r2 in self.basic_rules:
                try:
                    if all(self.check_match(r2(r1(np.array(ex['input']))), ex['output']) for ex in train_cases):
                        return lambda x, f1=r1, f2=r2: f2(f1(x))
                except: continue
        return None

    def solve(self, task):
        train = task['train']
        test = task['test']
        best_rule = self.find_best_rule(train)
        
        final_results = []
        for t in test:
            inp = np.array(t['input'])
            attempts = []

            # 1. Gunakan aturan terbaik jika ditemukan
            if best_rule:
                try:
                    res = best_rule(inp).tolist()
                    if res not in attempts: attempts.append(res)
                except: pass

            # 2. Fallback Heuristik (Identity & Color Shift paling sering benar di ARC)
            fallbacks = [self.identity(inp), self.color_shift(inp)]
            for fb in fallbacks:
                fb_list = fb.tolist()
                if fb_list not in attempts and len(attempts) < 2:
                    attempts.append(fb_list)

            # 3. Filler jika masih kurang dari 2
            while len(attempts) < 2:
                attempts.append(inp.tolist())

            final_results.append({"attempt_1": attempts[0], "attempt_2": attempts[1]})
        return final_results

# --- EXECUTION ---
import os

input_path = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json'
if not os.path.exists(input_path): # Local testing mode
    input_path = 'arc-agi_test_challenges.json'

try:
    with open(input_path, 'r') as f:
        tasks = json.load(f)
    
    solver = ARCSolver()
    submission = {tid: solver.solve(t) for tid, t in tasks.items()}

    with open('submission.json', 'w') as f:
        json.dump(submission, f)
    print("🚀 ULTIMATE SUBMISSION READY!")
except Exception as e:
    print(f"ERR: {e}")

🚀 ULTIMATE SUBMISSION READY!
